# RAG Agentik: Solusi Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari4/10-agentic-rag-solusi.ipynb)

Ini **solusi lengkap** semua latihan bab ini (2 latihan) di Google Colab. Semua bagian `TODO` sudah terisi, jadi notebook ini tinggal dijalankan (**Runtime > Run all**, atau `Shift+Enter` per cell). Keluaran dari satu kali eksekusi dan penjelasan tiap latihan sudah disertakan; jawaban model berubah tiap kali dijalankan, jadi hasil Anda bisa sedikit berbeda. Kerjakan dulu [notebook latihannya](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari4/10-agentic-rag-latihan.ipynb); buka solusi ini untuk mencocokkan hasil Anda atau saat macet.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Solusi Latihan 1: Asisten SOP dengan Uji Otomatis
Jalankan sel satu per satu dengan Shift+Enter

Asisten dokumen untuk operator lapangan, dengan enam uji yang diperiksa DI KODE:

1. Pengetahuan umum -> menjawab langsung, TIDAK mencari
2. Pertanyaan SOP -> mencari, menjawab dengan angka yang benar, mengutip SOP yang benar
3. Konflik revisi -> memakai Rev 3 (10 ppm), tidak memakai Rev 2 (20 ppm)
4. Kode alarm alat -> menemukan manual VSD dan menjelaskan kode F07
5. Tak terjawab -> menolak dengan kalimat standar
6. Dokumen terlarang -> menolak (operator tidak boleh membaca laporan investigasi HSE)

In [ ]:
# Langkah 1 - Menyiapkan agen
import os
import sys

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from migas_shared.config import ROOT
from migas_shared.rag_tools import KonteksPengguna

sys.path.insert(0, str(ROOT / "hari4" / "10-agentic-rag" / "code"))  # agar rag_agent.py ditemukan
from rag_agent import HasilRAG, bangun_agen_rag, tanya

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)
agent = bangun_agen_rag(model)
operator = KonteksPengguna(role="operator", area="Seruni", user_id="OP-114")

## Langkah 2 - Fungsi pemeriksa
`periksa` mencetak LOLOS atau GAGAL beserta rincian, lalu mengembalikan hasilnya.

In [ ]:
def periksa(nama: str, ok: bool, rincian: str) -> bool:
    print(f"{'LOLOS' if ok else 'GAGAL'}  {nama}")
    print(f"        {rincian}")
    return ok


hasil_uji: list[bool] = []

## Langkah 3 - Uji 1: pengetahuan umum

In [ ]:
r1: HasilRAG = tanya(agent, "Apa ibu kota Prancis?", operator)
hasil_uji.append(periksa("1. Pengetahuan umum dijawab langsung", not r1.mencari and "paris" in r1.jawaban.lower(),
                         f"mencari={r1.mencari}, jawaban: {r1.jawaban[:80]}"))

LOLOS  1. Pengetahuan umum dijawab langsung
        mencari=False, jawaban: Ibu kota Prancis adalah Paris.


## Langkah 4 - Uji 2: pertanyaan SOP

In [ ]:
r2 = tanya(agent, "Berapa nilai IDLH untuk gas H2S?", operator)
hasil_uji.append(periksa("2. Pertanyaan SOP: angka benar dan sitasi benar",
                         r2.mencari and "100 ppm" in r2.jawaban and "SOP-HSE-001" in r2.sitasi and not r2.sitasi_tidak_valid,
                         f"sitasi={r2.sitasi}, tidak valid={r2.sitasi_tidak_valid or 'tidak ada'}"))

LOLOS  2. Pertanyaan SOP: angka benar dan sitasi benar
        sitasi=['SOP-HSE-001'], tidak valid=tidak ada


## Langkah 5 - Uji 3: konflik revisi

In [ ]:
r3 = tanya(agent, "Pada konsentrasi H2S berapa pekerja wajib menghentikan pekerjaan dan evakuasi?", operator)
hasil_uji.append(periksa("3. Konflik revisi: memakai Rev 3 (10 ppm); Rev 2 tidak diambil",
                         "10 ppm" in r3.jawaban and all("superseded" not in x for x in r3.revisi_diambil),
                         f"revisi diambil={r3.revisi_diambil}"))

LOLOS  3. Konflik revisi: memakai Rev 3 (10 ppm); Rev 2 tidak diambil
        revisi diambil=['GLS-001 Rev 1 (active)', 'MAN-VSD-FD500 Rev 1.2 (active)', 'SOP-HSE-001 Rev 3 (active)']


## Langkah 6 - Uji 4: kode alarm alat

In [ ]:
r4 = tanya(agent, "Apa arti alarm F07 pada VSD FD-500 dan apakah unit boleh langsung di-restart?", operator)
hasil_uji.append(periksa("4. Kode alarm alat: manual VSD ditemukan",
                         "MAN-VSD-FD500" in r4.diambil and ("ground fault" in r4.jawaban.lower() or "f07" in r4.jawaban.lower()),
                         f"diambil={r4.diambil}"))

LOLOS  4. Kode alarm alat: manual VSD ditemukan
        diambil=['MAN-VSD-FD500', 'SOP-OPS-012']


## Langkah 7 - Uji 5: pertanyaan tak terjawab

In [ ]:
r5 = tanya(agent, "Siapa direktur utama PT Hulu Energi Nusantara?", operator)
hasil_uji.append(periksa("5. Pertanyaan tak terjawab ditolak", r5.menolak, f"jawaban: {r5.jawaban[:90]}"))

LOLOS  5. Pertanyaan tak terjawab ditolak
        jawaban: Informasi tidak ditemukan di dokumen yang tersedia.

Meskipun pencarian tidak menemukan in


## Langkah 8 - Uji 6: dokumen terlarang

In [ ]:
r6 = tanya(agent, "Apa akar masalah insiden paparan H2S di SR-027 pada Juli 2026?", operator)
hasil_uji.append(periksa("6. Dokumen terlarang tidak diambil dan ditolak",
                         "INV-2026-011" not in r6.diambil and r6.menolak,
                         f"diambil={r6.diambil}, menolak={r6.menolak}"))

LOLOS  6. Dokumen terlarang tidak diambil dan ditolak
        diambil=['RPT-SR-2026-07', 'SOP-HSE-001'], menolak=True


## Langkah 9 - Rekap

In [ ]:
print(f"Hasil: {sum(hasil_uji)}/{len(hasil_uji)} uji lolos")
print("(Uji berbasis model dapat sesekali gagal. Jalankan ulang dan analisis mengapa: salah cari, salah revisi, atau salah menolak?)")

Hasil: 6/6 uji lolos
(Uji berbasis model dapat sesekali gagal. Jalankan ulang dan analisis mengapa: salah cari, salah revisi, atau salah menolak?)


### Penjelasan Solusi Latihan 10.1: asisten SOP dengan uji otomatis

Agen RAG bisa memutuskan sendiri apakah perlu mencari dokumen. Karena itu mengujinya berarti memeriksa perilaku, bukan hanya membaca jawaban. Enam kasus diperiksa di kode.

- `TODO(B10.1)`: `bangun_agen_rag(model)` menyusun agen dengan tool `cari_dokumen`.
- `TODO(B10.2)`: pertanyaan SOP harus memicu pencarian, memuat "100 ppm" dan sitasi SOP-HSE-001, tanpa sitasi yang tidak ada di dokumen yang diambil.
- `TODO(B10.3)`: pada konflik revisi, jawaban harus "10 ppm" (Rev 3) dan tidak boleh ada revisi `superseded` di dokumen yang diambil.
- `TODO(B10.4)`: operator tidak boleh membaca laporan investigasi: `INV-2026-011` tidak boleh ada di dokumen yang diambil dan agen harus menolak.

Keenam uji lolos pada keluaran. Perhatikan yang tersirat: pertanyaan ibu kota Prancis dijawab tanpa mencari, pertanyaan direktur utama ditolak karena tidak ada di dokumen, dan akses dibatasi oleh filter peran pada pencarian, bukan oleh prompt, sehingga laporan investigasi memang tidak pernah masuk ke konteks. Uji berbasis model sesekali gagal. Kalau satu uji GAGAL, cari tahu apakah agen salah mencari, salah memilih revisi, atau salah menolak.

## Solusi Latihan Bonus: RAG Percakapan
Jalankan sel satu per satu dengan Shift+Enter

Menambahkan memori (bab 5) ke agen RAG agar pengguna bisa mengajukan follow-up question yang merujuk
jawaban sebelumnya, tanpa mengulang konteks:

- "Berapa lama fire watch harus berjaga setelah pekerjaan panas?" -> 30 menit
- "Dan bagaimana dengan batas LEL-nya?" -> (masih soal pekerjaan panas)
- "Dokumen mana yang tadi kamu rujuk?" -> menjawab dari riwayat, tanpa mencari

In [ ]:
# Langkah 1 - Menyiapkan agen dengan memori percakapan
import os
import re
import sys

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

from migas_shared.config import ROOT
from migas_shared.rag_tools import KonteksPengguna, cari_dokumen

sys.path.insert(0, str(ROOT / "hari4" / "10-agentic-rag" / "code"))  # agar rag_agent.py ditemukan
from rag_agent import PROMPT_RAG

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)
agent = create_agent(
    model,
    tools=[cari_dokumen],
    system_prompt=PROMPT_RAG + "\nPengguna boleh mengajukan pertanyaan lanjutan yang merujuk jawaban sebelumnya; pakai riwayat percakapan.",
    context_schema=KonteksPengguna,
    checkpointer=InMemorySaver(),
)
konteks = KonteksPengguna(role="operator", area="Seruni", user_id="OP-114")
config = {"configurable": {"thread_id": "sesi-operator-114"}}

## Langkah 2 - Fungsi satu giliran
`giliran` mengirim satu pertanyaan ke utas yang sama dan menghitung berapa kali agen mencari pada giliran itu.

In [ ]:
jumlah_pencarian = []


def giliran(pertanyaan: str) -> None:
    print(f"Pengguna: {pertanyaan}")
    sebelum = len(agent.get_state(config).values.get("messages", [])) if jumlah_pencarian else 0
    response = agent.invoke({"messages": [HumanMessage(content=pertanyaan)]}, config, context=konteks)
    baru = response["messages"][sebelum:]
    mencari = sum(1 for m in baru if isinstance(m, AIMessage) for p in m.tool_calls if p["name"] == "cari_dokumen")
    jumlah_pencarian.append(mencari)
    print(f"   Pencarian pada giliran ini: {mencari}")
    print(f"Agen: {response['messages'][-1].content}")

## Langkah 3 - Giliran 1: pertanyaan awal

In [ ]:
giliran("Berapa lama fire watch harus tetap berjaga setelah pekerjaan panas selesai?")

Pengguna: Berapa lama fire watch harus tetap berjaga setelah pekerjaan panas selesai?
   Pencarian pada giliran ini: 1
Agen: Fire watch harus tetap berjaga **minimal 30 menit setelah pekerjaan panas selesai** [SOP-HSE-004 5. Persyaratan Khusus Pekerjaan Panas (PTW-H)].


## Langkah 4 - Giliran 2: pertanyaan lanjutan
Pertanyaan ini bergantung pada konteks giliran sebelumnya.

In [ ]:
giliran("Dan bagaimana dengan batas LEL yang disyaratkan?")

Pengguna: Dan bagaimana dengan batas LEL yang disyaratkan?
   Pencarian pada giliran ini: 0
Agen: Berdasarkan dokumen yang ditemukan:

- **Uji gas sebelum mulai**: LEL harus **0%** [SOP-HSE-004 5. Persyaratan Khusus Pekerjaan Panas (PTW-H)].
- **Pemantauan gas kontinu selama pekerjaan**: Jika **LEL ≥ 5%**, pekerjaan harus segera dihentikan [SOP-HSE-004 5. Persyaratan Khusus Pekerjaan Panas (PTW-H)].


## Langkah 5 - Giliran 3: merujuk riwayat

In [ ]:
giliran("Dokumen apa yang tadi kamu rujuk untuk dua jawaban itu?")

Pengguna: Dokumen apa yang tadi kamu rujuk untuk dua jawaban itu?
   Pencarian pada giliran ini: 0
Agen: Dua jawaban tersebut sama-sama saya rujuk dari dokumen **SOP-HSE-004** revisi 5 (status active), bagian **"5. Persyaratan Khusus Pekerjaan Panas (PTW-H)"**.

Sitasi lengkapnya: [SOP-HSE-004 5. Persyaratan Khusus Pekerjaan Panas (PTW-H)].


## Langkah 6 - Pemeriksaan di kode

In [ ]:
status = agent.get_state(config)
print(f"Utas menyimpan {len(status.values['messages'])} pesan (pertanyaan, pencarian, hasil, dan jawaban).")
teks_akhir = str(status.values["messages"][-1].content)
ok_ingatan = bool(re.search(r"SOP-HSE-004|LL-2026-03", teks_akhir))
print(f"{'LOLOS' if ok_ingatan else 'GAGAL'}  pertanyaan ke-3 menyebut dokumen dari giliran sebelumnya (SOP-HSE-004)")
print(f"{'LOLOS' if jumlah_pencarian[0] >= 1 else 'GAGAL'}  pertanyaan pertama mencari di dokumen")

Utas menyimpan 8 pesan (pertanyaan, pencarian, hasil, dan jawaban).
LOLOS  pertanyaan ke-3 menyebut dokumen dari giliran sebelumnya (SOP-HSE-004)
LOLOS  pertanyaan pertama mencari di dokumen


### Penjelasan Solusi Latihan 10.2: RAG percakapan

Memori dari bab 5 ditambahkan ke agen RAG, sehingga pertanyaan lanjutan tidak perlu mengulang konteks.

- `TODO(B10.5)`: agen dengan `cari_dokumen`, `PROMPT_RAG` yang ditambah izin pertanyaan lanjutan, `context_schema=KonteksPengguna`, dan `InMemorySaver`.
- `TODO(B10.6)`: setiap giliran memakai `config` thread yang sama dan `context=konteks`.
- `TODO(B10.7)`: jumlah pencarian dihitung dari `AIMessage` pada pesan baru yang memanggil `cari_dokumen`.

Hasil yang terlihat pada keluaran: giliran pertama mencari satu kali (fire watch 30 menit). Giliran kedua, pertanyaan tentang batas LEL, dijawab tanpa mencari lagi karena bagian SOP yang sama sudah ada di riwayat. Giliran ketiga ("dokumen apa yang tadi kamu rujuk") juga tanpa pencarian dan menyebut SOP-HSE-004. Satu detail perlu dicurigai: jawaban itu menyebut "revisi 5", padahal angka 5 pada sitasi adalah nomor bagian dokumen, bukan revisi. Agen salah membaca format sitasi. Itu contoh mengapa sitasi sebaiknya diperiksa di kode, seperti di latihan sebelumnya.